<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Deep-Neural-Network-with-Hyperparameter-tuning-Regularization-and-Optimization-from-scrach/blob/main/Deep%20Neural%20Network%20with%20Hyperparameter%20tuning%2C%20Regularization%20and%20Optimization%20from%20scrach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles

X, Y = make_circles(n_samples=6000,noise=0.15,factor=0.5)

In [74]:

print(Y.shape[0] , X.T.shape)


6000 (2, 6000)


In [75]:
print(X.shape[1])

2


In [76]:
def init_parameters(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [77]:
def init_parameters_he(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * np.sqrt(2/layers_dims[l-1])
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [78]:
def init_parameters_random(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.random.randn(layers_dims[l] , layers_dims[l-1]) * 0.01
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b


In [79]:
def init_parameters_zeros(dim):
  layers_dims = dim
  L = len(layers_dims)
  W = {}
  b = {}
  for l in range(1 , L):
    W[l] = np.zeros((layers_dims[l] , layers_dims[l-1]))
    b[l] = np.zeros((layers_dims[l] , 1))
  return W , b

In [80]:
def forward(W , b , X , L):
  Zs = []
  As = []
  A = X.T
  As.append(A)

  def relu(Z):
    return np.maximum(0 , Z)
  def sigmoid(Z):
    return 1/(1 + np.exp(-Z))

  for l in range(1 , L-1):
    A_prev = A
    Z = np.dot(W[l],A_prev)+b[l]
    Zs.append(Z)
    A = relu(Z)
    As.append(A)

  ZL = np.dot(W[L-1],A)+b[L-1]
  AL = sigmoid(ZL)
  Zs.append(ZL)
  As.append(AL)
  return Zs , As

In [81]:
def cost(Y , Y_hat):
  m = Y.shape[1]
  temp = Y * np.log(Y_hat) + (1-Y) * np.log(1-Y_hat)
  return -np.sum(temp)/m

In [82]:
def backward(W, Zs, As, Y, L):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T)
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [83]:
def backward_with_L2(W, Zs, As, Y, L , lambd):

    dW = {}

    db = {}

    m = Y.shape[1]

    AL = As[-1]

    dZ = AL - Y

    for l in reversed(range(1, L)):
        dW[l] = (1/m) * np.dot(dZ, As[l-1].T) + (lambd/m)*W[l]
        db[l] = (1/m) * np.sum(dZ,axis=1,keepdims=True)
        if l > 1:
            dA = np.dot(W[l].T, dZ)
            dZ = dA * (Zs[l-2] > 0)

    return dW, db

In [84]:
def gradient(W , b , X , Y , L , alpha):
  Zs , As = forward(W , b , X , L)
  dW , db = backward(W , Zs , As , Y , L)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [85]:
def gradient_with_L2(W , b , X , Y , L , alpha , lambd):
  Zs , As = forward(W , b , X , L)
  dW , db = backward_with_L2(W , Zs , As , Y , L , lambd)
  for l in range(1, L):
    W[l] = W[l] - alpha * dW[l]
    b[l] = b[l] - alpha * db[l]
  return W , b

In [86]:
def L2_cost(W, b, X, Y, L, lambd):

    m = X.shape[1]

    L2 = 0

    for l in range(1, L):
        L2 += np.sum(W[l]**2)

    _, As = forward(W, b, X, L)
    cross = cost(Y, As[-1])
    reg = (lambd/(2*m))*L2

    # print("Cross Entropy:", cross)
    # print("L2 Penalty:", reg)

    # return cross + reg

    return cost(Y, As[-1]) + (lambd/(2*m))*L2

In [87]:
def model(dim , X , Y , initialization , reg , alpha):

  if initialization == "zeros":
        W , b = init_parameters_zeros(dim)
  elif initialization == "random":
        W , b = init_parameters_random(dim)
  elif initialization == "he":
        W , b = init_parameters_he(dim)

  # W , b = init_parameters(dim)
  L = len(dim)

  for i in range(10000):
    if(reg == "L2"):
      W , b = gradient_with_L2(W , b , X , Y , L , alpha , 0.7)
    elif(reg == "normal"):
      W , b = gradient(W , b , X , Y , L , alpha)
    if i % 1000 == 0:
        _, As = forward(W, b, X, L)
        if(reg == "L2"):
          c = L2_cost(W, b, X, Y, L, 0.7)
        elif(reg == "normal"):
          c = cost(Y, As[-1])
        print(f"Iteration {i}: Cost = {c:.4f}")
  return W , b

optimization techniques


In [88]:
def batch_model(X , Y ,  epochs , batch_size , dim):
  # epochs = 1000
  # batch_size = 64
  # dim = [2 , 3 , 1]
  L = len(dim)
  W, b = init_parameters_he(dim)
  if Y.ndim == 1:
    Y = Y.reshape(1, -1)

  def random_mini_batches(X, Y, batch_size=64, seed=None):

      if seed is not None:
          np.random.seed(seed)

      m = X.shape[0]
      mini_batches = []

      permutation = np.random.permutation(m)

      X_shuffled = X[permutation, :]
      Y_shuffled = Y[:, permutation]

      num_complete_batches = m // batch_size

      for k in range(num_complete_batches):
          start = k * batch_size
          end = start + batch_size

          X_batch = X_shuffled[start:end, :]
          Y_batch = Y_shuffled[:, start:end]
          mini_batches.append((X_batch, Y_batch))


      if m % batch_size != 0:
          start = num_complete_batches * batch_size
          X_batch = X_shuffled[start:m, :]
          Y_batch = Y_shuffled[:, start:m]
          mini_batches.append((X_batch, Y_batch))

      return mini_batches

  for epoch in range(epochs):

      mini_batches = random_mini_batches(X, Y, batch_size)

      epoch_cost = 0

      for X_batch, Y_batch in mini_batches:

          W , b = gradient(W , b , X_batch , Y_batch , L , 0.01)

          _, As_batch = forward(W, b, X_batch, L)
          batch_cost = cost(Y_batch, As_batch[-1])
          epoch_cost += batch_cost

      epoch_cost /= len(mini_batches)

      if epoch % 100 == 0:
          print(f"Epoch {epoch}: Cost = {epoch_cost:.5f}")
  return W , b

Mini-Batch-gradient-descent fn ==> batch_model()


In [89]:
L = len([2 , 3 , 1])
W , b =  batch_model(X , Y , 1000 , 64 , [2 , 3 , 1])
Zs, As = forward(W, b, X, L)
predictions = (As[-1] >= 0.5)
accuracy = np.mean(predictions == Y) * 100
print(f"Accuracy: {accuracy:.2f}%")

Epoch 0: Cost = 0.69804
Epoch 100: Cost = 0.29217
Epoch 200: Cost = 0.21063
Epoch 300: Cost = 0.18277
Epoch 400: Cost = 0.16761
Epoch 500: Cost = 0.15829
Epoch 600: Cost = 0.15175
Epoch 700: Cost = 0.14736
Epoch 800: Cost = 0.14412
Epoch 900: Cost = 0.14118
Accuracy: 95.10%


In [90]:
def initialize_velocity(W, b):
    vdw = {}
    vdb = {}
    L = len(W) + 1

    for l in range(1, L):
        vdw[l] = np.zeros_like(W[l])
        vdb[l] = np.zeros_like(b[l])
    return vdw, vdb
def update_with_momentum(W, b, dW, db, vdw, vdb, beta, alpha):
    L = len(W) + 1
    for l in range(1, L):
      vdw[l] = beta * vdw[l] + (1 - beta) * dW[l]
      vdb[l] = beta * vdb[l] + (1 - beta) * db[l]

      W[l] = W[l] - alpha * vdw[l]
      b[l] = b[l] - alpha * vdb[l]

    return W, b, vdw, vdb

In [91]:
def momentum(X , Y ,  epochs , batch_size , dim , Beta , alpha ):

  # epochs = 1000
  # batch_size = 64
  # dim = [2 , 3 , 1]
  L = len(dim)
  W, b = init_parameters_he(dim)
  if Y.ndim == 1:
    Y = Y.reshape(1, -1)

  def random_mini_batches(X, Y, batch_size=64, seed=None):

      if seed is not None:
          np.random.seed(seed)

      m = X.shape[0]
      mini_batches = []

      permutation = np.random.permutation(m)

      X_shuffled = X[permutation, :]
      Y_shuffled = Y[:, permutation]

      num_complete_batches = m // batch_size

      for k in range(num_complete_batches):
          start = k * batch_size
          end = start + batch_size

          X_batch = X_shuffled[start:end, :]
          Y_batch = Y_shuffled[:, start:end]
          mini_batches.append((X_batch, Y_batch))


      if m % batch_size != 0:
          start = num_complete_batches * batch_size
          X_batch = X_shuffled[start:m, :]
          Y_batch = Y_shuffled[:, start:m]
          mini_batches.append((X_batch, Y_batch))

      return mini_batches

  vdw, vdb = initialize_velocity(W, b)

  for epoch in range(epochs):

      mini_batches = random_mini_batches(X, Y, batch_size)

      epoch_cost = 0

      for X_batch, Y_batch in mini_batches:

          Zs , As = forward(W , b , X_batch , L)
          dW, db = backward(W, Zs, As, Y_batch, L)
          W , b , vdw , vdb = update_with_momentum(W, b, dW, db, vdw, vdb, Beta, alpha)

          _, As_batch = forward(W, b, X_batch, L)
          batch_cost = cost(Y_batch, As_batch[-1])
          epoch_cost += batch_cost

      epoch_cost /= len(mini_batches)

      if epoch % 100 == 0:
          print(f"Epoch {epoch}: Cost = {epoch_cost:.5f}")
  return W , b




momentum(X , Y ,  epochs , batch_size , dim , Beta , alpha )

In [92]:
W , b = momentum(X , Y ,  1000 , 64 , [2 , 3 , 1] , 0.9 , 0.01 )
Zs, As = forward(W, b, X, L)
predictions = (As[-1] >= 0.5)
accuracy = np.mean(predictions == Y) * 100
print(f"Accuracy: {accuracy:.2f}%")

Epoch 0: Cost = 0.67151
Epoch 100: Cost = 0.27573
Epoch 200: Cost = 0.20564
Epoch 300: Cost = 0.17940
Epoch 400: Cost = 0.16584
Epoch 500: Cost = 0.15728
Epoch 600: Cost = 0.15175
Epoch 700: Cost = 0.14756
Epoch 800: Cost = 0.14455
Epoch 900: Cost = 0.14189
Accuracy: 95.03%


In [93]:
def initialize_velocity_adam(W, b):
    vdw = {}
    vdb = {}
    sdw = {}
    sdb = {}
    L = len(W) + 1

    for l in range(1, L):
        sdw[l] = np.zeros_like(W[l])
        sdb[l] = np.zeros_like(b[l])
        vdw[l] = np.zeros_like(W[l])
        vdb[l] = np.zeros_like(b[l])
    return vdw, vdb , sdw , sdb
def update_with_momentum_adam(W, b, dW, db, vdw, vdb, sdw , sdb , beta, alpha , eps , t):
    L = len(W) + 1
    for l in range(1, L):
      vdw[l] = beta * vdw[l] + (1 - beta) * dW[l]
      vdb[l] = beta * vdb[l] + (1 - beta) * db[l]
      sdw[l] = beta * sdw[l] + (1 - beta) * dW[l]**2
      sdb[l] = beta * sdb[l] + (1 - beta) * db[l]**2
      vdw_corrected = vdw[l] / (1 - beta**t)
      vdb_corrected = vdb[l] / (1 - beta**t)
      sdw_corrected = sdw[l] / (1 - beta**t)
      sdb_corrected = sdb[l] / (1 - beta**t)

      W[l] = W[l] - alpha * vdw_corrected/np.sqrt(sdw_corrected + eps)
      b[l] = b[l] - alpha * vdb_corrected/np.sqrt(sdb_corrected + eps)

    return W, b, vdw, vdb , sdw , sdb

In [94]:
def adam(X , Y ,  epochs , batch_size , dim , Beta , alpha , eps):

  # epochs = 1000
  # batch_size = 64
  # dim = [2 , 3 , 1]
  L = len(dim)
  W, b = init_parameters_he(dim)
  if Y.ndim == 1:
    Y = Y.reshape(1, -1)

  def random_mini_batches(X, Y, batch_size=64, seed=None):

      if seed is not None:
          np.random.seed(seed)

      m = X.shape[0]
      mini_batches = []

      permutation = np.random.permutation(m)

      X_shuffled = X[permutation, :]
      Y_shuffled = Y[:, permutation]

      num_complete_batches = m // batch_size

      for k in range(num_complete_batches):
          start = k * batch_size
          end = start + batch_size

          X_batch = X_shuffled[start:end, :]
          Y_batch = Y_shuffled[:, start:end]
          mini_batches.append((X_batch, Y_batch))


      if m % batch_size != 0:
          start = num_complete_batches * batch_size
          X_batch = X_shuffled[start:m, :]
          Y_batch = Y_shuffled[:, start:m]
          mini_batches.append((X_batch, Y_batch))

      return mini_batches

  vdw, vdb , sdw , sdb = initialize_velocity_adam(W, b)
  t = 0
  for epoch in range(epochs):

      mini_batches = random_mini_batches(X, Y, batch_size)

      epoch_cost = 0

      for X_batch, Y_batch in mini_batches:
          t += 1
          Zs , As = forward(W , b , X_batch , L)
          dW, db = backward(W, Zs, As, Y_batch, L)
          W , b , vdw , vdb , sdw , sdb = update_with_momentum_adam(W , b , dW , db , vdw , vdb , sdw , sdb , Beta , alpha , eps , t)

          _, As_batch = forward(W, b, X_batch, L)
          batch_cost = cost(Y_batch, As_batch[-1])
          epoch_cost += batch_cost

      epoch_cost /= len(mini_batches)

      if epoch % 100 == 0:
          print(f"Epoch {epoch}: Cost = {epoch_cost:.5f}")
  return W , b




adam(X , Y ,  epochs , batch_size , dim , Beta , alpha , eps)

In [98]:
W , b = adam(X , Y ,  1000 , 64 , [2 , 3 , 1] , 0.9 , 0.001 , 10e-8)
Zs, As = forward(W, b, X, L)
predictions = (As[-1] >= 0.5)
accuracy = np.mean(predictions == Y) * 100
print(f"Accuracy: {accuracy:.2f}%")

Epoch 0: Cost = 0.67550
Epoch 100: Cost = 0.20906
Epoch 200: Cost = 0.15592
Epoch 300: Cost = 0.14205
Epoch 400: Cost = 0.13655
Epoch 500: Cost = 0.13379
Epoch 600: Cost = 0.13181
Epoch 700: Cost = 0.13059
Epoch 800: Cost = 0.13002
Epoch 900: Cost = 0.12964
Accuracy: 95.22%
